In [1]:
# ==========================
# === CHANGE EPISODE PIPELINE (CLEAN "LIST OF EPISODES" FORMAT) ===
# Outputs (under Observations):
#   1) List_Change_Episodes.csv
#   2) List_Boundary_Commit_Events.csv
#   3) Repo_level_episodes.csv
#
# EPISODE LOGIC (clean):
#   - Episode START = an EFFECTIVE "added" event (style was not active before)
#   - Episode END   = min( next EFFECTIVE "added" event (any style),
#                          first EFFECTIVE "removed" event of the start style )
#                    tie-break: prefer "removed" (now enforced by key ordering)
#   - Keep episodes that are >= PERSIST_DAYS OR active at cutoff.
#
# FIXES APPLIED:
#   - FIX #1: event key includes etype order (removed before added) to enforce tie-break
#   - FIX #2: env_styles is RAW (start style only) — no "A || B" contamination
#   - FIX #3: boundary_event_types includes end boundary type when episode ends via an event
# ==========================

from __future__ import annotations

import json
from pathlib import Path
from datetime import datetime, timezone
from typing import Dict, List, Optional, Tuple, Set
from collections import defaultdict
import bisect

import pandas as pd

# ---- Paths ----
WORK_ROOT   = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2")
OUTPUT_MINE = WORK_ROOT / "Mine_Full"

OBS_OUTPUT = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2\Observations")
OBS_OUTPUT.mkdir(parents=True, exist_ok=True)

# ---- Settings ----
CUTOFF_ISO   = "2025-08-10 23:59:59 +0000"
PERSIST_DAYS = 14.0

STYLE_ORDER: List[str] = ["Emu_Custom", "Emu_Community", "GMD", "ThirdParty"]
STYLE_RANK: Dict[str, int] = {s: i for i, s in enumerate(STYLE_ORDER)}
CANONICAL_STYLES: Set[str] = set(STYLE_ORDER)

# --------------------------
# Helpers
# --------------------------
def parse_iso(s: Optional[str]) -> Optional[datetime]:
    if not s:
        return None
    s = str(s).strip()
    if s.endswith("Z"):
        s = s[:-1] + "+00:00"
    # normalize +0000 -> +00:00
    if len(s) >= 5 and (s[-5] in ["+", "-"]) and s[-3] != ":":
        s = s[:-2] + ":" + s[-2:]
    try:
        dt = datetime.fromisoformat(s)
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return dt.astimezone(timezone.utc)
    except Exception:
        return None

CUTOFF_DT = parse_iso(CUTOFF_ISO)
assert CUTOFF_DT is not None, f"Bad CUTOFF_ISO: {CUTOFF_ISO}"

def days_between(a: datetime, b: datetime) -> float:
    return (b - a).total_seconds() / 86400.0

def read_repo_jsons(output_dir: Path) -> List[Path]:
    return sorted(output_dir.glob("*.emulator_timeline*.json"))

def repo_from_filename(jf: Path) -> str:
    name = jf.name
    if ".emulator_timeline" in name:
        return name.split(".emulator_timeline", 1)[0]
    return jf.stem

def is_default_scoped(jf: Path, rec: Dict) -> bool:
    scope = str(rec.get("timeline_scope") or "").lower()
    nm = jf.name.lower()
    return ("default_first_parent" in scope) or ("default_refs" in scope) or ("default_refs" in nm) or ("default_first_parent" in nm)

def default_scope_score(jf: Path, rec: Dict) -> int:
    scope = str(rec.get("timeline_scope") or "").strip().lower()
    nm = jf.name.lower()
    if "default_first_parent" in scope or "default_first_parent" in nm:
        return 2
    if "default_refs" in scope or "default_refs" in nm:
        return 1
    return 0

def iso(dt: datetime) -> str:
    return dt.astimezone(timezone.utc).isoformat()

# --------------------------
# Select ONE default-scoped JSON per repo (prefer default_first_parent)
# --------------------------
json_files = read_repo_jsons(OUTPUT_MINE)
if not json_files:
    raise FileNotFoundError(f"No miner JSON files found under: {OUTPUT_MINE}")

best_by_repo: Dict[str, Tuple[int, Path, Dict]] = {}  # repo -> (score, path, rec)

for jf in json_files:
    try:
        with jf.open("r", encoding="utf-8") as f:
            rec = json.load(f)
    except Exception:
        continue

    repo = rec.get("repo_name") or repo_from_filename(jf)
    if not repo:
        continue

    if not is_default_scoped(jf, rec):
        continue

    score = default_scope_score(jf, rec)
    if score <= 0:
        continue

    if repo not in best_by_repo or score > best_by_repo[repo][0]:
        best_by_repo[repo] = (score, jf, rec)

selected = list(best_by_repo.values())
selected.sort(key=lambda t: str(t[1].name).lower())

print(f"[info] Found {len(json_files)} miner JSON files total.")
print(f"[info] Selected {len(selected)} DEFAULT-scoped repos (prefer default_first_parent).")

# --------------------------
# Build EFFECTIVE events (only those that actually flip active status)
# --------------------------
def collect_effective_events(rec: Dict, cutoff_dt: datetime) -> Tuple[List[Dict], Dict[str, List[Dict]], List[Dict]]:
    """
    Returns:
      start_events: list of effective 'added' events (style was not active before)
      removals_by_style: style -> list of effective 'removed' events
      all_effective: all effective events (added/removed) in time order
    """
    events_dict = rec.get("events") or {}
    raw: List[Dict] = []

    for style in STYLE_ORDER:
        for ev in events_dict.get(style, []):
            et = str(ev.get("event") or "").strip().lower()
            if et not in {"added", "removed"}:
                continue

            dt = parse_iso(ev.get("date"))
            if dt is None or dt > cutoff_dt:
                continue

            sha = (ev.get("commit") or "").strip()
            if not sha:
                continue

            # default-scoped JSONs: treat missing on_default as 1
            on_default = ev.get("on_default", 1)
            if str(on_default) != "1":
                continue

            raw.append({
                "dt": dt.astimezone(timezone.utc),
                "commit": sha,
                "style": style,
                "etype": et,  # 'added' or 'removed'
            })

    # Deterministic processing:
    # - sort by date, commit
    # - same date+commit: removals first, then additions
    # - then stable style ordering
    raw.sort(key=lambda e: (
        e["dt"],
        e["commit"],
        0 if e["etype"] == "removed" else 1,
        STYLE_RANK.get(e["style"], 99),
    ))

    active: Set[str] = set()
    start_events: List[Dict] = []
    removals_by_style: Dict[str, List[Dict]] = {s: [] for s in STYLE_ORDER}
    all_effective: List[Dict] = []

    for e in raw:
        s = e["style"]
        et = e["etype"]

        # FIX #1: key includes etype order so tie-break truly prefers removed
        etype_order = 0 if et == "removed" else 1
        key = (e["dt"], e["commit"], etype_order, STYLE_RANK.get(s, 99))

        if et == "removed":
            if s in active:
                active.remove(s)
                eff = {**e, "key": key}
                removals_by_style[s].append(eff)
                all_effective.append(eff)
        else:  # added
            if s not in active:
                active.add(s)
                eff = {**e, "key": key}
                start_events.append(eff)
                all_effective.append(eff)

    return start_events, removals_by_style, all_effective

# --------------------------
# Build CLEAN change episodes from start_events
# --------------------------
def build_clean_change_episodes(
    start_events: List[Dict],
    removals_by_style: Dict[str, List[Dict]],
    cutoff_dt: datetime,
    persist_days: float,
) -> Tuple[List[Dict], List[Dict]]:
    """
    Builds episodes:
      - start = effective 'added' event
      - end   = min(next start_event, first removal of start_style), tie -> removal
      - end_commit blank if active at cutoff
    Returns: (raw_episodes, kept_episodes)
    """
    if not start_events:
        return [], []

    start_events = sorted(start_events, key=lambda e: e["key"])

    # Precompute removal keys for bisect
    rem_keys: Dict[str, List[Tuple]] = {}
    for s, lst in removals_by_style.items():
        lst.sort(key=lambda r: r["key"])
        rem_keys[s] = [r["key"] for r in lst]

    raw_eps: List[Dict] = []

    for i, se in enumerate(start_events):
        s_key = se["key"]
        s_dt: datetime = se["dt"]
        s_commit: str = se["commit"]
        s_style: str = se["style"]

        # Candidate A: next start event (any style)
        next_start = start_events[i + 1] if (i + 1) < len(start_events) else None

        # Candidate B: first removal of the same start style after this start
        r_list = removals_by_style.get(s_style, [])
        r_k = rem_keys.get(s_style, [])
        j = bisect.bisect_right(r_k, s_key)
        next_rem = r_list[j] if j < len(r_list) else None

        # Choose end event (tie -> removal, now correctly enforced by key ordering)
        end_event = None
        if next_start is not None and next_rem is not None:
            end_event = next_rem if next_rem["key"] <= next_start["key"] else next_start
        elif next_start is not None:
            end_event = next_start
        elif next_rem is not None:
            end_event = next_rem

        if end_event is None:
            e_dt = cutoff_dt
            e_commit = ""
            end_style = ""
            end_type = ""
        else:
            e_dt = end_event["dt"]
            e_commit = end_event["commit"]
            end_style = end_event["style"]
            end_type = end_event["etype"]  # 'added' or 'removed'

        if e_dt <= s_dt:
            continue

        dur = max(0.0, days_between(s_dt, e_dt))
        is_active_at_cutoff = 1 if (end_event is None and e_dt == cutoff_dt) else 0

        # FIX #2: env_styles should be RAW (episode style), not "start || end"
        env_styles = s_style

        # FIX #3: boundary_event_types should include BOTH boundaries when episode ends by an event
        boundary_types = "added" if end_event is None else f"added || {end_type}"

        # boundary dates: single if active; "start || end" otherwise
        boundary_dates = iso(s_dt) if end_event is None else f"{iso(s_dt)} || {iso(e_dt)}"

        ep = {
            "episode_start_dt": s_dt,
            "episode_end_dt": e_dt,

            "episode_start_commit_sha": s_commit,
            "episode_end_commit_sha": e_commit,

            "env_styles": env_styles,
            "boundary_event_types": boundary_types,
            "boundary_event_dates": boundary_dates,

            "episode_duration_days": dur,
            "is_active_at_cutoff": is_active_at_cutoff,
        }
        raw_eps.append(ep)

    # Keep episodes that are >= persist_days OR active at cutoff
    kept = [
        ep for ep in raw_eps
        if (ep["episode_duration_days"] >= persist_days) or (ep["episode_end_dt"] == cutoff_dt and ep["episode_end_commit_sha"] == "")
    ]
    kept.sort(key=lambda ep: ep["episode_start_dt"])

    return raw_eps, kept

# --------------------------
# Main pipeline
# --------------------------
episode_rows: List[Dict] = []
boundary_event_rows: List[Dict] = []
repo_level_rows: List[Dict] = []

for score, jf, rec in selected:
    repo = rec.get("repo_name") or repo_from_filename(jf)
    if not repo:
        continue

    # Match your expected formatting: Owner__Repo and Owner.Repo
    repo_name = repo
    full_name = repo.replace("__", ".")

    timeline_scope = str(rec.get("timeline_scope", "") or "")
    qa_issue = str(rec.get("qa_issue", "") or "")

    start_events, removals_by_style, all_effective = collect_effective_events(rec, CUTOFF_DT)
    raw_eps, kept_eps = build_clean_change_episodes(start_events, removals_by_style, CUTOFF_DT, PERSIST_DAYS)

    # Assign episode_index AFTER filtering
    for idx, ep in enumerate(kept_eps, start=1):
        ep_row = {
            "repo_name": repo_name,
            "full_name": full_name,
            "episode_index": idx,

            "episode_start_utc": iso(ep["episode_start_dt"]),
            "episode_end_utc": iso(ep["episode_end_dt"]),
            "episode_duration_days": f"{ep['episode_duration_days']:.6f}",

            "env_styles": ep["env_styles"],  # RAW style now (e.g., "GMD")
            "boundary_event_types": ep["boundary_event_types"],  # e.g., "added || added" or "added || removed"
            "boundary_event_dates": ep["boundary_event_dates"],

            "episode_start_commit_sha": ep["episode_start_commit_sha"],
            "episode_end_commit_sha": ep["episode_end_commit_sha"],
        }
        episode_rows.append(ep_row)

    # ----- Optional: boundary events dataset (only those used by kept episode boundaries)
    kept_start_commits = {ep["episode_start_commit_sha"] for ep in kept_eps}
    kept_end_commits = {ep["episode_end_commit_sha"] for ep in kept_eps if ep["episode_end_commit_sha"]}

    start_commit_to_idx = defaultdict(list)
    end_commit_to_idx = defaultdict(list)
    for idx, ep in enumerate(kept_eps, start=1):
        start_commit_to_idx[ep["episode_start_commit_sha"]].append(idx)
        if ep["episode_end_commit_sha"]:
            end_commit_to_idx[ep["episode_end_commit_sha"]].append(idx)

    for ev in all_effective:
        sha = ev["commit"]
        if sha not in kept_start_commits and sha not in kept_end_commits:
            continue

        is_start = 1 if sha in kept_start_commits else 0
        is_end = 1 if sha in kept_end_commits else 0

        assigned_ep = ""
        if is_start:
            assigned_ep = str(min(start_commit_to_idx[sha]))
        elif is_end:
            assigned_ep = str(max(end_commit_to_idx[sha]))

        boundary_event_rows.append({
            "repo_name": repo_name,
            "full_name": full_name,
            "timeline_scope": timeline_scope,
            "qa_issue": qa_issue,

            "env_style": ev["style"],
            "event_type": ev["etype"],
            "event_date_utc": iso(ev["dt"]),
            "commit_sha": sha,

            "episode_index": assigned_ep,
            "is_episode_start_boundary": is_start,
            "is_episode_end_boundary": is_end,
        })

    # ----- Optional: repo-level summary
    first_ep = kept_eps[0] if kept_eps else None
    current_ep = next((ep for ep in kept_eps if ep["episode_end_dt"] == CUTOFF_DT and ep["episode_end_commit_sha"] == ""), None)

    repo_level_rows.append({
        "repo_name": repo_name,
        "full_name": full_name,
        "cutoff_date": CUTOFF_ISO,
        "persist_days": PERSIST_DAYS,
        "timeline_scope": timeline_scope,
        "qa_issue": qa_issue,

        "episodes_total_raw": len(raw_eps),
        "episodes_total_kept": len(kept_eps),

        "first_episode_start_utc": iso(first_ep["episode_start_dt"]) if first_ep else "",
        "first_episode_env_styles": first_ep["env_styles"] if first_ep else "",
        "first_episode_start_commit_sha": first_ep["episode_start_commit_sha"] if first_ep else "",

        "current_episode_start_utc": iso(current_ep["episode_start_dt"]) if current_ep else "",
        "current_episode_env_styles": current_ep["env_styles"] if current_ep else "",
        "current_episode_start_commit_sha": current_ep["episode_start_commit_sha"] if current_ep else "",
        "current_episode_duration_days": f"{days_between(current_ep['episode_start_dt'], CUTOFF_DT):.6f}" if current_ep else "",
    })

# --------------------------
# Write List_Change_Episodes.csv
# --------------------------
episodes_df = pd.DataFrame.from_records(episode_rows)
if not episodes_df.empty:
    episodes_df.sort_values(["repo_name", "episode_index"], inplace=True)

episodes_out_csv = OBS_OUTPUT / "List_Change_Episodes.csv"
episodes_df.to_csv(episodes_out_csv, index=False, encoding="utf-8")
print(f"[ok] Wrote CLEAN episode dataset: {episodes_out_csv}")

# --------------------------
# Write List_Boundary_Commit_Events.csv (optional)
# --------------------------
events_df = pd.DataFrame.from_records(boundary_event_rows)
if not events_df.empty:
    events_df.sort_values(["repo_name", "episode_index", "event_date_utc", "env_style", "event_type"], inplace=True)

rq4_out_csv = OBS_OUTPUT / "List_Boundary_Commit_Events.csv"
events_df.to_csv(rq4_out_csv, index=False, encoding="utf-8")
print(f"[ok] Wrote boundary-commit dataset: {rq4_out_csv}")

# --------------------------
# Write Repo_level_episodes.csv (optional)
# --------------------------
repo_df = pd.DataFrame.from_records(repo_level_rows)
if not repo_df.empty:
    repo_df.sort_values(["repo_name"], inplace=True)

repo_out_csv = OBS_OUTPUT / "Repo_level_episodes.csv"
repo_df.to_csv(repo_out_csv, index=False, encoding="utf-8")
print(f"[ok] Wrote repo-level summary: {repo_out_csv}")


[info] Found 480 miner JSON files total.
[info] Selected 480 DEFAULT-scoped repos (prefer default_first_parent).
[ok] Wrote CLEAN episode dataset: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2\Observations\List_Change_Episodes.csv
[ok] Wrote boundary-commit dataset: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2\Observations\List_Boundary_Commit_Events.csv
[ok] Wrote repo-level summary: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2\Observations\Repo_level_episodes.csv
